**Create table on BigQuery**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Scans"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# SCHEMA
# =========================================

schema = [

    bigquery.SchemaField(
        "Scan_ID",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Patient_U_ID",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Scan_Date",
        "DATE"
    ),

    bigquery.SchemaField(
        "Scan_Code",
        "STRING"
    ),

    bigquery.SchemaField(
        "Scan_Group",
        "STRING"
    ),

    bigquery.SchemaField(
        "Scan_Name",
        "STRING"
    ),

    bigquery.SchemaField(
        "Scan_Findings_Conc",
        "STRING"
    ),

    bigquery.SchemaField(
        "Scan_Path_Link",
        "STRING"
    )

]

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

table.clustering_fields = [
    "Patient_U_ID",
    "Scan_Code"
]

client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("BigQuery table created successfully")
print(TABLE_REF)
print("=================================")

BigQuery table created successfully
depihealthnux.Depihealthnux_Main.Scans


**Create Table on Postgres**

In [2]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS scan_record_no_seq;

""")

# =========================================
# CREATE TABLE
# =========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS Scans (

    Scan_ID TEXT PRIMARY KEY
    DEFAULT (
        'SI_' ||
        LPAD(
            nextval('scan_record_no_seq')::TEXT,
            4,
            '0'
        )
    ),

    Patient_U_ID TEXT NOT NULL,

    Scan_Date DATE,

    Scan_Code TEXT NOT NULL,

    Scan_Findings_Conc TEXT,

    Scan_Path_Link TEXT

);
"""

cursor.execute(create_table_query)

# =========================================
# INDEXES
# =========================================

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_scans_patient
ON Scans(Patient_U_ID);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_scans_code
ON Scans(Scan_Code);

""")

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Scans")
print("=================================")

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Scans


**Sync BigQuery to Postgres**

In [3]:
import sys
import pandas as pd
import psycopg2

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ BIGQUERY
# =========================================

query = """

SELECT

    Scan_ID,
    Patient_U_ID,
    Scan_Date,
    Scan_Code,
    Scan_Findings_Conc,
    Scan_Path_Link

FROM `depihealthnux.Depihealthnux_Main.Scans`

ORDER BY Scan_ID

"""

df = client.query(query).to_dataframe()

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

# =========================================
# CONNECT POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""
DELETE FROM Scans;
""")

if not df.empty:

    execute_values(
        cursor,
        """
        INSERT INTO Scans (

            Scan_ID,
            Patient_U_ID,
            Scan_Date,
            Scan_Code,
            Scan_Findings_Conc,
            Scan_Path_Link

        )
        VALUES %s
        """,
        df.values.tolist(),
        page_size=1000
    )

# =========================================
# RESET SEQUENCE
# =========================================

cursor.execute("""

SELECT setval(
    'scan_record_no_seq',
    COALESCE(
        (
            SELECT MAX(
                CAST(
                    REPLACE(Scan_ID,'SI_','')
                    AS INTEGER
                )
            )
            FROM Scans
        ),
        1
    ),
    true
);

""")

conn.commit()

print(f"Inserted {len(df)} rows")

cursor.execute("""

SELECT *
FROM Scans
ORDER BY Scan_ID
LIMIT 3

""")

print("\nFirst 3 Rows From PostgreSQL")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

Empty DataFrame
Columns: [Scan_ID, Patient_U_ID, Scan_Date, Scan_Code, Scan_Findings_Conc, Scan_Path_Link]
Index: []
Rows Retrieved: 0
Inserted 0 rows

First 3 Rows From PostgreSQL


**Sync Postgress to BigQuery**

In [6]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT

    s.Scan_ID,
    s.Patient_U_ID,
    s.Scan_Date,

    s.Scan_Code,

    r.Scan_Group,
    r.Scan_Name,

    s.Scan_Findings_Conc,
    s.Scan_Path_Link

FROM Scans s

LEFT JOIN Scans_Ref r
    ON s.Scan_Code = r.Scan_Code

ORDER BY s.Scan_ID

"""

df = pd.read_sql(query, conn)

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn.close()

# =========================================
# LOAD TO BIGQUERY
# =========================================

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    df,
    "depihealthnux.Depihealthnux_Main.Scans",
    job_config=job_config
)

job.result()

print(f"Uploaded {len(df)} rows")

# =========================================
# VERIFY
# =========================================

verify_df = client.query("""

SELECT *
FROM `depihealthnux.Depihealthnux_Main.Scans`
LIMIT 3

""").to_dataframe()

print("\nFirst 3 Rows From BigQuery")

print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_4816\2829593004.py:55: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


   scan_id patient_u_id   scan_date scan_code scan_group   scan_name  \
0  SI_0001   PAT_000001  2026-05-20     SC001         US      US_Abd   
1  SI_0002   PAT_000001  2026-05-20     SC002         US  US_Thyroid   
2  SI_0003   PAT_000001  2026-05-20     SC003         US  US_Cranial   

  scan_findings_conc                              scan_path_link  
0             Normal  Scanhx/PAT_000001/Brain_MRI_2026_05_20.pdf  
1             Normal   Scanhx/PAT_000001/Chest_CT_2026_05_21.pdf  
2             Normal                                        None  
Rows Retrieved: 6
Uploaded 6 rows

First 3 Rows From BigQuery
   scan_id patient_u_id   scan_date scan_code scan_group   scan_name  \
0  SI_0001   PAT_000001  2026-05-20     SC001         US      US_Abd   
1  SI_0002   PAT_000001  2026-05-20     SC002         US  US_Thyroid   
2  SI_0003   PAT_000001  2026-05-20     SC003         US  US_Cranial   

  scan_findings_conc                              scan_path_link  
0             Normal  Sca